# 3-1. Advanced RAG: Query Rewriting & Multi-query

## 학습 목표

- 사용자 질문을 검색에 유리한 표현으로 재작성한다.
- 하나의 질문을 여러 개의 검색 질문으로 확장한다.
- 검색 누락을 줄이는 방법을 이해한다.

## 공통 전제

- 실습 데이터: `../data/공직자_민원응대_핵심_매뉴얼.pdf`
- 기본 모델: `gpt-4o-mini`
- 기본 임베딩 모델: `text-embedding-3-small`
- 벡터DB: Qdrant 로컬 인메모리 모드

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../../.env")

True

## 1. 기본 RAG 데이터 준비 완료 
- vector db 재사용
- `6.modular_rag_manual/notebooks/2-2_vector_embedding_store_qdrant_md.ipynb`

## 2. 규칙 기반 Query Rewriting

처음에는 LLM을 쓰지 않고, 키워드 확장 방식으로 단순하게 구현한다.

In [2]:
def rewrite_query(question: str) -> str:
    rules = [
        (["반복", "계속 전화", "같은 말"], "정당한 사유 없는 반복전화 동일민원 통화 종료 상담 종료"),
        (["장시간", "오래", "권장시간"], "정당한 사유 없는 장시간 통화 권장시간 상담 종료"),
        (["상급자", "기관장", "부서장"], "상급자 통화 요구 실무자 대응 담당 팀장 대응"),
        (["욕설", "폭언", "협박", "성희롱"], "폭언 욕설 협박 성희롱 증거 확보 통화 종료 법적조치"),
        (["폭행", "때리"], "폭행 비상대응팀 경찰 신고 피해공무원 분리"),
        (["위험물", "흉기", "칼"], "위험물 소지 경찰 신고 119 신고 대피"),
    ]

    for keywords, expansion in rules:
        if any(keyword in question for keyword in keywords):
            return f"{question}\n검색 키워드: {expansion}"

    return question


question = "민원인이 계속 같은 말을 반복하면서 전화하면 어떻게 해야 하나요?"

rewritten_question = rewrite_query(question)

print("[원 질문]")
print(question)

print("\n[재작성 질문]")
print(rewritten_question)

[원 질문]
민원인이 계속 같은 말을 반복하면서 전화하면 어떻게 해야 하나요?

[재작성 질문]
민원인이 계속 같은 말을 반복하면서 전화하면 어떻게 해야 하나요?
검색 키워드: 정당한 사유 없는 반복전화 동일민원 통화 종료 상담 종료


## 3. 원 질문 검색과 재작성 질문 검색 비교

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "civil_complaint_manual_medium"

EMBEDDING_MODEL = "text-embedding-3-small"

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

vector_store = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    url=QDRANT_URL,
)

In [4]:
original_docs = vector_store.similarity_search(question, k=3)
rewritten_docs = vector_store.similarity_search(rewritten_question, k=3)

print("[원 질문 검색 결과]")
for doc in original_docs:
    print(doc.metadata, doc.page_content[:250].replace("\n", " "))

print("\n[재작성 질문 검색 결과]")
for doc in rewritten_docs:
    print(doc.metadata, doc.page_content[:250].replace("\n", " "))

[원 질문 검색 결과]
{'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md', 'page': 5, 'section': '일반적인 민원 응대요령', 'topic': '일반 민원응대', 'document_type': 'preprocessed_markdown', 'chunk_type': 'page', 'chunk_id': 5, 'chunk_size': 700, 'chunk_overlap': 120, 'chunk_strategy': 'medium', 'embedding_model': 'text-embedding-3-small', 'collection_name': 'civil_complaint_manual_medium', '_id': '9911c641-e7e9-4cf1-b830-593e18236e6b', '_collection_name': 'civil_complaint_manual_medium'} STEP - 민원인에게 인사한 후, 소속 등을 밝힘 STEP - 민원인의 문의사항을 정확하게 판단하여 상담 진행 STEP - 민원상담이 끝나면 인사 후에 전화 종료 ※ 전화로 접수·처리가 불가능한 민원을 제기한 경우에는 서면 등(인터넷·팩스 등 정보통신망 또는 우편)으로 제출하거나 방문하도록 유도 민원인을 배려하는 표현 무심코 쓰는 표현 민원인을 배려하는 표현 ‣ 기다리세요. ‣ 모르겠는데요. ‣ 할 수 없는데요. ‣ 자리에 없습니다. ‣ 담당
{'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md', 'page': 7, 'section': '일반적인 민원 응대요령', 'topic': '장시간 통화', 'document_type': 'preprocessed_markdown', 'chunk_type': 'page', 'chunk_id': 8, 'chunk_size': 700, 'chunk_overlap': 120, 'chunk_strategy': 'medium', 'embedding_model': 'text-embed

## 4. Multi-query Retrieval

하나의 질문을 여러 검색 질문으로 나누어 검색한다.

In [5]:
multi_queries = [
    question,
    "정당한 사유 없는 반복전화 대응 방법",
    "동일민원 3회 이상 반복 제기 통화 종료",
    "반복전화 상담 종료 부서장 보고 휴게시간 부여"
]

collected_docs = []
seen = set()

for q in multi_queries:
    docs = vector_store.similarity_search(q, k=2)

    for doc in docs:
        key = (doc.metadata.get("page"), doc.page_content[:80])
        if key not in seen:
            seen.add(key)
            collected_docs.append(doc)

print("통합 검색 결과 수:", len(collected_docs))

for doc in collected_docs:
    print(doc.metadata, doc.page_content[:300].replace("\n", " "))
    print()

통합 검색 결과 수: 4
{'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md', 'page': 5, 'section': '일반적인 민원 응대요령', 'topic': '일반 민원응대', 'document_type': 'preprocessed_markdown', 'chunk_type': 'page', 'chunk_id': 5, 'chunk_size': 700, 'chunk_overlap': 120, 'chunk_strategy': 'medium', 'embedding_model': 'text-embedding-3-small', 'collection_name': 'civil_complaint_manual_medium', '_id': '9911c641-e7e9-4cf1-b830-593e18236e6b', '_collection_name': 'civil_complaint_manual_medium'} STEP - 민원인에게 인사한 후, 소속 등을 밝힘 STEP - 민원인의 문의사항을 정확하게 판단하여 상담 진행 STEP - 민원상담이 끝나면 인사 후에 전화 종료 ※ 전화로 접수·처리가 불가능한 민원을 제기한 경우에는 서면 등(인터넷·팩스 등 정보통신망 또는 우편)으로 제출하거나 방문하도록 유도 민원인을 배려하는 표현 무심코 쓰는 표현 민원인을 배려하는 표현 ‣ 기다리세요. ‣ 모르겠는데요. ‣ 할 수 없는데요. ‣ 자리에 없습니다. ‣ 담당자가 아니라서 모르겠는데요. ‣ 예? 뭐라고요? ‣ 아닙니다. **12** [ | ][현장

{'source': '공직자_민원응대_핵심_매뉴얼_rag_page_chunks.md', 'page': 7, 'section': '일반적인 민원 응대요령', 'topic': '장시간 통화', 'document_type': 'preprocessed_markdown', 'chunk_type': 'page', 'chunk_id': 8, 'chunk_size': 700, 'chunk_overlap': 120, 'chunk

## 5. LLM 기반 Multi-query 생성

실제 서비스에서는 LLM에게 검색 질문을 생성하게 할 수도 있다.

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

query_prompt = ChatPromptTemplate.from_messages([
    ("system", """
사용자의 질문을 벡터 검색에 적합한 한국어 검색 질문 3개로 바꾸세요.
각 줄에 하나의 검색 질문만 출력하세요.
"""),
    ("human", "{question}")
])

query_chain = query_prompt | llm | StrOutputParser()

# LLM이 검색 질문 3개 생성
generated_queries_text = query_chain.invoke({"question": question})
print(generated_queries_text)

# 줄 단위로 파싱해 리스트로 변환
generated_queries = [
    line.strip("- ").strip()
    for line in generated_queries_text.splitlines()
    if line.strip()
]

print(generated_queries)


민원인이 반복적으로 같은 말을 할 때 대처 방법은?  
전화로 민원인이 같은 말을 반복할 때 어떻게 대응해야 하나요?  
민원인과의 전화 통화에서 반복적인 질문에 대한 해결책은 무엇인가요?
['민원인이 반복적으로 같은 말을 할 때 대처 방법은?', '전화로 민원인이 같은 말을 반복할 때 어떻게 대응해야 하나요?', '민원인과의 전화 통화에서 반복적인 질문에 대한 해결책은 무엇인가요?']


## 핵심 정리

- Query Rewriting은 사용자의 자연어 질문을 검색에 유리한 표현으로 바꾸는 기법이다.
- Multi-query Retrieval은 하나의 질문을 여러 검색 질문으로 확장해 검색 누락을 줄인다.
- Modular RAG에서는 Query Rewriting을 독립 모듈로 분리할 수 있다.